# Querying the PIM read API from Python

This notebook is a hands-on tour of the read-only FastAPI service that serves the
classified NASA PIM whitelists (**instruments**, **platforms**, **missions**).

You'll see how to:

1. Check readiness (`/health`)
2. Fetch a page of records and read the pagination envelope
3. Paginate through an entire dataset with a generator
4. Filter by PIM type and by science division
5. Use `ETag` / `If-None-Match` for conditional GETs (`304 Not Modified`)
6. Load results into a pandas DataFrame for quick analysis
7. Handle validation errors (`422`)

Everything uses [`requests`](https://requests.readthedocs.io/) (already a project
dependency). pandas is optional — the analysis cell degrades gracefully if it's
not installed.

## Prerequisites — pick an API instance

Connection details are read from the repo's `.env` file (see `.env.example`):

| variable            | purpose                                                        |
| ------------------- | -------------------------------------------------------------- |
| `PIM_API_BASE_URL`  | base URL of the API (defaults to `http://localhost:8000`)      |
| `PIM_API_KEY`       | `x-api-key` for the deployed CloudFront endpoint (optional locally) |

**Deployed:** `.env` ships pointed at the CloudFront endpoint, which requires the
`x-api-key` header — nothing else to do.

**Local instead:** run the API from the repo root and set
`PIM_API_BASE_URL=http://localhost:8000` (no key needed):

```bash
uv sync
uv run uvicorn pim_whitelist.api.app:app --reload --port 8000
```

That serves the whitelists from `whitelist/classified/` (Swagger UI at `/docs`).

In [2]:
import os
import requests
from dotenv import find_dotenv, load_dotenv

# Load PIM_API_BASE_URL / PIM_API_KEY from the repo's .env. usecwd=True walks up
# from the working directory, so this finds the repo-root .env whether Jupyter
# was launched from the repo root or from examples/.
load_dotenv(find_dotenv(usecwd=True))

BASE_URL = os.environ.get("PIM_API_BASE_URL", "http://localhost:8000").rstrip("/")
API_KEY = os.environ.get("PIM_API_KEY")

# One Session reuses the TCP connection across requests and holds shared headers,
# including the x-api-key that the deployed CloudFront endpoint requires.
session = requests.Session()
session.headers.update({"Accept": "application/json"})
if API_KEY:
    session.headers["x-api-key"] = API_KEY

print(f"Talking to {BASE_URL}")
print("x-api-key:", "set" if API_KEY else "not set (fine for localhost)")

Talking to https://d2vzyrjsfbe2fn.cloudfront.net
x-api-key: set


## A tiny typed client

A thin wrapper keeps the rest of the notebook readable: it joins the path, raises
on HTTP errors, and returns parsed JSON. We also keep the raw `Response` around
for the ETag examples later.

In [3]:
def get(path, **params):
    """GET {BASE_URL}{path} with optional query params; return (json, response).

    Raises requests.HTTPError on 4xx/5xx so mistakes surface loudly.
    """
    # Drop params that are None so we don't send empty query values.
    params = {k: v for k, v in params.items() if v is not None}
    resp = session.get(f"{BASE_URL}{path}", params=params, timeout=30)
    resp.raise_for_status()
    return resp.json(), resp

## 1. Readiness probe — `GET /health`

`/health` returns `200` once every dataset has loaded at least one record, else
`503` with per-type counts. It's unauthenticated and un-paginated — a good first
call to confirm the service is up and pointed at real data.

In [4]:
health, resp = get("/health")
print("status:", resp.status_code)
health

status: 200


{'status': 'ok',
 'data_version': 'cfeba4ebb14130a8',
 'counts': {'instrument': 7352, 'platform': 4384, 'mission': 1901}}

## 2. Fetch a page — `GET /fetch_pims_records`

Every list endpoint returns the same **pagination envelope**:

| field         | meaning                                   |
| ------------- | ----------------------------------------- |
| `items`       | the records on this page                  |
| `page`        | current page (1-based)                    |
| `page_size`   | records per page (50–100, default 50)     |
| `total`       | total records matching the query          |
| `total_pages` | number of pages at this `page_size`       |

Each record looks like:

```json
{
  "canonical_name": "MISR",
  "aliases": ["Multi-Angle Imaging SpectroRadiometer"],
  "division": ["earth"],
  "source": "SDE",
  "type": "instrument"
}
```

In [5]:
page, _ = get("/fetch_pims_records", page=1, page_size=50)

print(f"page {page['page']} of {page['total_pages']}  |  {page['total']} total records")
print(f"{len(page['items'])} items on this page\n")

for rec in page["items"][:5]:
    divs = ", ".join(rec["division"]) or "—"
    print(f"[{rec['type']:<10}] {rec['canonical_name']}  ({divs}, {rec['source']})")

page 1 of 273  |  13637 total records
50 items on this page

[instrument] 10cm refractor full-disk white-light imaging telescope  (heliophysics, SDE)
[instrument] 120-COLOR CIRCULAR-VARIABLE-FILTER (CVF) PHOTOMETER  (planetary, SDE)
[instrument] 120 Color Circular Variable Filter Photometer  (astrophysics, LLM)
[instrument] 1.3GHz transportable wind profiler radar (called LQ4) at Koganei, Japan.  (heliophysics, SDE)
[instrument] 1D Particles Probe  (earth, LLM)


## 3. Paginate through everything

A generator that walks pages until `page == total_pages` lets you stream every
record without holding more than one page in memory at a time. Ask for the max
`page_size` (100) to minimize round trips.

In [6]:
def iter_records(path="/fetch_pims_records", *, division=None, page_size=100):
    """Yield every record from a list endpoint, following pagination."""
    page_num = 1
    while True:
        page, _ = get(path, page=page_num, page_size=page_size, division=division)
        yield from page["items"]
        if page_num >= page["total_pages"]:
            break
        page_num += 1


all_records = list(iter_records())
print(f"pulled {len(all_records)} records total")

from collections import Counter
by_type = Counter(r["type"] for r in all_records)
print("by type:", dict(by_type))

pulled 13637 records total
by type: {'instrument': 7352, 'platform': 4384, 'mission': 1901}


## 4a. Filter by PIM type

Each type has its own endpoint. The response shape is identical — only the
contents differ (and every item's `type` matches the endpoint).

| endpoint                              | contents      |
| ------------------------------------- | ------------- |
| `/fetch_pims_records/instruments`     | instruments   |
| `/fetch_pims_records/platforms`       | platforms     |
| `/fetch_pims_records/missions`        | missions      |

In [7]:
missions, _ = get("/fetch_pims_records/missions", page=1, page_size=100)
print(f"{missions['total']} missions\n")
for m in missions["items"][:8]:
    print("•", m["canonical_name"])

1901 missions

• 1M/1
• 1M/2
• 2001 Mars Odyssey
• 3D Winds
• 3MI
• Able 1
• Able 2
• ABRIXAS


## 4b. Filter by science division

The `division` query param narrows any list endpoint to one of:
`earth`, `heliophysics`, `planetary`, `astrophysics`, `bps`. It composes with
the type endpoints and with pagination.

In [8]:
earth_instruments = list(iter_records(
    "/fetch_pims_records/instruments", division="earth"))

print(f"{len(earth_instruments)} Earth-science instruments\n")
for r in earth_instruments[:8]:
    print("•", r["canonical_name"])

2043 Earth-science instruments

• 1D Particles Probe
• 2B Technologies Nitric Oxide Monitor
• 2B Technologies Nitrogen Dioxide Converter
• 2B Technologies Ozone Monitor
• 2D Cloud Probe
• 2D-Gray
• 2DS
• 2D Stereo Particle Probe


## 5. Conditional GET with `ETag` → `304 Not Modified`

Every response carries an `ETag` (derived from the data version + query) and a
`Cache-Control` header. Send the ETag back as `If-None-Match` and, if nothing
changed, the server returns `304` with an empty body — letting a client or edge
cache skip re-downloading the page.

> Note: `raise_for_status()` treats `304` as success (it's not a 4xx/5xx), so we
> call the endpoint directly here to inspect the status code.

In [9]:
# First request: capture the ETag.
first = session.get(f"{BASE_URL}/fetch_pims_records/platforms", timeout=30)
etag = first.headers["ETag"]
print("1st GET :", first.status_code, "| ETag:", etag)
print("Cache-Control:", first.headers.get("Cache-Control"))

# Second request with If-None-Match: expect 304 and an empty body.
second = session.get(
    f"{BASE_URL}/fetch_pims_records/platforms",
    headers={"If-None-Match": etag},
    timeout=30,
)
print("2nd GET :", second.status_code, "| body bytes:", len(second.content))

1st GET : 200 | ETag: "cfeba4ebb14130a8-platform-*-1-50"
Cache-Control: public, max-age=300
2nd GET : 304 | body bytes: 0


## 6. Into pandas for quick analysis

The flat records drop straight into a DataFrame. `division` is a list per record
(a PIM can belong to more than one division), so we `explode` it to count
records per division. This cell is a no-op message if pandas isn't installed.

In [10]:
try:
    import pandas as pd

    df = pd.DataFrame(all_records)
    display(df.head())

    print("\nRecords per type:")
    print(df["type"].value_counts())

    print("\nRecords per division (a record can span several):")
    print(df.explode("division")["division"].value_counts())

    print("\nRecords per source:")
    print(df["source"].value_counts())
except ModuleNotFoundError:
    print("pandas not installed — `uv pip install pandas` to run this cell.")

,canonical_name,aliases,division,source,type
0,10cm refractor full-disk white-light imaging t...,[10cm refractor full-disk white-light imaging ...,[heliophysics],SDE,instrument
1,120-COLOR CIRCULAR-VARIABLE-FILTER (CVF) PHOTO...,[120-COLOR CIRCULAR-VARIABLE-FILTER (CVF) PHOT...,[planetary],SDE,instrument
2,120 Color Circular Variable Filter Photometer,[120 Color Circular Variable Filter Photometer...,[astrophysics],LLM,instrument
3,1.3GHz transportable wind profiler radar (call...,[1.3GHz transportable wind profiler radar (cal...,[heliophysics],SDE,instrument
4,1D Particles Probe,"[1D Particles Probe, 1DP]",[earth],LLM,instrument



Records per type:
type
instrument    7352
platform      4384
mission       1901
Name: count, dtype: int64

Records per division (a record can span several):
division
heliophysics    6769
earth           4708
planetary       1675
bps              513
astrophysics     344
Name: count, dtype: int64

Records per source:
source
SDE    7228
LLM    6409
Name: count, dtype: int64


## 7. Validation errors — `422`

Out-of-range or unknown query values are rejected with `422 Unprocessable
Entity` and a body describing the problem. Examples: `page=0` (must be ≥ 1),
`page_size=10` (must be 50–100), `division=lunar` (not a known division).

In [11]:
bad = session.get(
    f"{BASE_URL}/fetch_pims_records",
    params={"page_size": 10},  # below the minimum of 50
    timeout=30,
)
print("status:", bad.status_code)
print(bad.json())

status: 422
{'detail': [{'type': 'greater_than_equal', 'loc': ['query', 'page_size'], 'msg': 'Input should be greater than or equal to 50', 'input': '10', 'ctx': {'ge': 50}}]}


## Where to go next

- Browse the interactive schema at `BASE_URL/docs` (Swagger UI) or `/redoc`.
- See `docs/read-api-architecture.md` for how the index is built and served.
- The same envelope + ETag contract holds on the deployed Lambda instance —
  just change `PIM_API_BASE_URL` at the top.